<a href="https://colab.research.google.com/github/pranayar/Job-search-tool/blob/main/Jobs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlencode
from tabulate import tabulate

keywords = "software developer OR .NET OR C# NOT senior NOT manager NOT architect NOT principal NOT Leader NOT Lead NOT Senior"
location = "Toronto, Ontario, Canada"

params = {
    "keywords": keywords,
    "location": location,
    "f_TPR": "r86400",  # last 24 hours,
     "f_TPR": "r604800"  # last 7 days

}

url = "https://www.linkedin.com/jobs/search/?" + urlencode(params)

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

rows = []

for job in soup.select(".base-card"):
    title_elem = job.select_one(".base-search-card__title")
    company_elem = job.select_one(".base-search-card__subtitle")
    location_elem = job.select_one(".job-search-card__location")
    link_elem = job.select_one("a.base-card__full-link")

    title = title_elem.get_text(strip=True) if title_elem else ""
    company = company_elem.get_text(strip=True) if company_elem else ""
    location = location_elem.get_text(strip=True) if location_elem else ""
    link = link_elem.get("href", "") if link_elem else ""

    if company not in ("Astra-North Infoteck Inc.  ~ Conquering today’s challenges, achieving tomorrow’s vision!","Mercor",):
      rows.append([
        company,
        title,
        location,
        link
      ])

print(
    tabulate(
        rows,
        headers=[
            "Company Name",
            "Job Role",
            "Location",
            "Job Link"
        ],
        tablefmt="grid"
    )
)

+-------------------------+------------------------------------------------------------+----------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| Company Name            | Job Role                                                   | Location                                           | Job Link                                                                                                                                                                                                                  |
+=========================+============================================================+====================================================+=======================================================================================================================================

In [13]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlencode
from tabulate import tabulate

# Search criteria
keywords = "software developer OR .NET OR C#"
location = "Toronto, Ontario, Canada"

# Jobs must contain at least one of these
REQUIRED_WORDS = {
    "developer",
    ".net",
    "c#",
    "software"
}

# Jobs containing any of these will be skipped
EXCLUDED_WORDS = {
    "senior",
    "sr.",
    "sr ",
    "lead",
    "leader",
    "manager",
    "architect",
    "principal",
    "director",
    "staff",
    "vp",
    "vice president"
}

# Companies to ignore
EXCLUDED_COMPANIES = {
    "Mercor",
    "Astra-North Infoteck Inc.  ~ Conquering today’s challenges, achieving tomorrow’s vision!"
}

params = {
    "keywords": keywords,
    "location": location,
    "f_TPR": "r604800"  # Last 7 days
}

url = "https://www.linkedin.com/jobs/search/?" + urlencode(params)

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

rows = []
seen = set()

for job in soup.select(".base-card"):
    title_elem = job.select_one(".base-search-card__title")
    company_elem = job.select_one(".base-search-card__subtitle")
    location_elem = job.select_one(".job-search-card__location")
    link_elem = job.select_one("a.base-card__full-link")
    time_elem = job.select_one("time")

    title = title_elem.get_text(strip=True) if title_elem else ""
    company = company_elem.get_text(strip=True) if company_elem else ""
    job_location = location_elem.get_text(strip=True) if location_elem else ""
    link = link_elem.get("href", "") if link_elem else ""
    posted = time_elem.get_text(strip=True) if time_elem else ""

    # Skip unwanted companies
    if company in EXCLUDED_COMPANIES:
        continue

    search_text = f"{title} {company}".lower()

    # Must contain at least one required keyword
    if not any(word.lower() in search_text for word in REQUIRED_WORDS):
        continue

    # Skip unwanted roles
    if any(word.lower() in search_text for word in EXCLUDED_WORDS):
        continue

    # Remove duplicates
    key = (company.lower(), title.lower())
    if key in seen:
        continue

    seen.add(key)

    rows.append([
        company,
        title,
        job_location,
        posted,
        link
    ])

if rows:
    print(
        tabulate(
            rows,
            headers=[
                "Company Name",
                "Job Role",
                "Location",
                "Posted",
                "Job Link"
            ],
            tablefmt="grid",
            # maxcolwidths=[30, 40, 25, 15, 60]
        )
    )
    print(f"\nTotal jobs found: {len(rows)}")
else:
    print("No matching jobs found.")

+---------------------------------+------------------------------------------------------+--------------------------------+--------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| Company Name                    | Job Role                                             | Location                       | Posted       | Job Link                                                                                                                                                                                                                  |
+=================================+======================================================+================================+==============+================================================================================================================================================